# 06 — Business impact in the buyer's KPIs (GRR, churned ARR, CSM capacity)

Converts the out-of-time model evaluation (`gold_model_metrics`, T-90 checkpoint) into the KPIs
the VP Customer Success and the CFO track. All inputs are read from tables produced earlier in the
job; assumptions are explicit parameters so the business can challenge them.

**Save-rate assumptions (conservative, adjustable):**
* A save play started ≥ 90 days before renewal (exec sponsor call, adoption workshop, sensor
  re-provisioning, right-sizing) retains **30%** of at-risk ARR.
* A save play started only when the customer sends a cancellation ticket (median 31 days before
  renewal, status quo) retains **10%**.

In [2]:
import json
import pandas as pd

dbutils.widgets.text("catalog", "ytcy_azure_east2classic_stable")
dbutils.widgets.text("schema", "officepulse_churn")
dbutils.widgets.text("save_rate_early", "0.30")
dbutils.widgets.text("save_rate_late", "0.10")
FQ = f"{dbutils.widgets.get('catalog')}.{dbutils.widgets.get('schema')}"
SAVE_EARLY = float(dbutils.widgets.get("save_rate_early"))
SAVE_LATE = float(dbutils.widgets.get("save_rate_late"))

m = {r["metric"]: r["value"] for r in spark.table(f"{FQ}.gold_model_metrics").collect()}
display(spark.table(f"{FQ}.gold_model_metrics").orderBy("metric_group", "metric"))

| metric | value | metric_group |
|---|---|---|
| ml_threshold | 0.4 | config |
| model_version | 5.0 | config |
| rules_threshold_same_volume | 9.077806477732794 | config |
| base_rate | 0.1905476369092273 | model_eval |
| lift_at_top10pct | 4.005076668048074 | model_eval |
| precision_at_top10pct | 0.7631578947368421 | model_eval |
| precision_at_top20pct | 0.575984990619137 | model_eval |
| precision_at_top5pct | 0.8120300751879699 | model_eval |
| recall_at_top10pct | 0.39960629921259844 | model_eval |
| recall_at_top20pct | 0.6043307086614174 | model_eval |
| recall_at_top5pct | 0.2125984251968504 | model_eval |
| rules_auc | 0.7716020046266228 | model_eval |
| rules_pr_auc | 0.39298007785333255 | model_eval |
| rules_precision_at_top10pct | 0.40225563909774437 | model_eval |
| rules_precision_at_top20pct | 0.4071294559099437 | model_eval |
| rules_precision_at_top5pct | 0.40601503759398494 | model_eval |
| rules_recall_at_top10pct | 0.2106299212598425 | model_eval |
| rules_recal

 ## Trailing-12-month churn, ARR base and today's risk book

In [4]:
# Trailing-12-month churn and ARR base from the Lakeflow gold tables
t12 = spark.sql(f"""
SELECT sum(churned_accounts) AS churned_logos_12m, sum(churned_arr_jpy) AS churned_arr_12m
FROM {FQ}.gold_churn_monthly WHERE month >= add_months(DATE'2026-11-01', -12) AND month < DATE'2026-11-01'
""").first()
base = spark.sql(f"""
SELECT sum(CASE WHEN c.start_date <= add_months(DATE'2026-11-01', -12)
                 AND (c.end_date IS NULL OR c.end_date > add_months(DATE'2026-11-01', -12)) THEN a.arr_jpy END) AS arr_start_12m,
       sum(CASE WHEN c.end_date IS NULL THEN a.arr_jpy END) AS arr_live_now
FROM {FQ}.silver_contracts c JOIN {FQ}.silver_accounts a USING (account_id)
""").first()
now = spark.sql(f"""
SELECT count(*) AS scored, sum(CASE WHEN risk_band = '高' THEN 1 ELSE 0 END) AS high,
       sum(CASE WHEN risk_band = '高' THEN arr_jpy ELSE 0 END) AS high_arr,
       sum(CASE WHEN hidden_risk THEN 1 ELSE 0 END) AS hidden, sum(CASE WHEN hidden_risk THEN arr_jpy ELSE 0 END) AS hidden_arr,
       sum(arr_at_risk_jpy) AS expected_arr_at_risk,
       sum(CASE WHEN risk_band = '高' AND datediff(renewal_date, DATE'2026-11-02') BETWEEN 0 AND 180 THEN arr_jpy ELSE 0 END) AS high_arr_renewing_180d
FROM {FQ}.gold_account_360 WHERE churn_probability IS NOT NULL
""").first()

print(t12, base, now, sep="\n")

Row(churned_logos_12m=93, churned_arr_12m=329910000)
Row(arr_start_12m=2144320000, arr_live_now=2190690000)
Row(scored=448, high=41, high_arr=119810000, hidden=36, hidden_arr=101870000, expected_arr_at_risk=280038415.0, high_arr_renewing_180d=96200000)


 ## Business case

In [6]:
churned_arr = float(t12.churned_arr_12m)
arr_start = float(base.arr_start_12m)
ml_cov = m["ml_arr_recall_at_t90"]
rules_cov = m["rules_arr_recall_at_t90"]
ticket_cov = m["cancel_ticket_share_of_churners"]

saved_status_quo = churned_arr * ticket_cov * SAVE_LATE
saved_rules = churned_arr * rules_cov * SAVE_EARLY
saved_ml = churned_arr * ml_cov * SAVE_EARLY
grr_now = 1 - churned_arr / arr_start
impact = {
    "arr_live_now_jpy": float(base.arr_live_now),
    "arr_start_12m_jpy": arr_start,
    "churned_logos_12m": float(t12.churned_logos_12m),
    "churned_arr_12m_jpy": churned_arr,
    "gross_revenue_retention_12m": grr_now,
    "ml_arr_coverage_t90": ml_cov,
    "rules_arr_coverage_t90": rules_cov,
    "cancel_ticket_coverage_any_time": ticket_cov,
    "arr_saved_status_quo_jpy": saved_status_quo,
    "arr_saved_with_ml_jpy": saved_ml,
    "incremental_arr_saved_vs_status_quo_jpy": saved_ml - saved_status_quo,
    "incremental_arr_saved_vs_rules_jpy": saved_ml - saved_rules,
    # observed churn already includes the status-quo saves, so only the increment improves GRR
    "grr_with_ml": 1 - (churned_arr - (saved_ml - saved_status_quo)) / arr_start,
    "grr_uplift_pts": (saved_ml - saved_status_quo) / arr_start * 100,
    "alerts_per_quarter_t90": m["ml_alerts_at_t90"] / 2,   # test window covers ~6 months of renewals
    "alert_precision_t90": m["ml_precision_at_t90"],
    "accounts_scored_now": float(now.scored),
    "high_risk_accounts_now": float(now.high),
    "high_risk_arr_now_jpy": float(now.high_arr),
    "high_risk_arr_renewing_180d_jpy": float(now.high_arr_renewing_180d),
    "hidden_risk_accounts_now": float(now.hidden),
    "hidden_risk_arr_now_jpy": float(now.hidden_arr),
    "expected_arr_at_risk_now_jpy": float(now.expected_arr_at_risk),
    "save_rate_early": SAVE_EARLY,
    "save_rate_late": SAVE_LATE,
}
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
display(pd.DataFrame(impact.items(), columns=["kpi", "value"]))

| kpi | value |
|---|---|
| arr_live_now_jpy | 2190690000.0 |
| arr_start_12m_jpy | 2144320000.0 |
| churned_logos_12m | 93.0 |
| churned_arr_12m_jpy | 329910000.0 |
| gross_revenue_retention_12m | 0.846147030293986 |
| ml_arr_coverage_t90 | 0.6059684774781983 |
| rules_arr_coverage_t90 | 0.5385230818465477 |
| cancel_ticket_coverage_any_time | 0.46808510638297873 |
| arr_saved_status_quo_jpy | 15442595.744680852 |
| arr_saved_with_ml_jpy | 59974518.12144972 |
| incremental_arr_saved_vs_status_quo_jpy | 44531922.376768865 |
| incremental_arr_saved_vs_rules_jpy | 6675273.141851351 |
| grr_with_ml | 0.8669144168672441 |
| grr_uplift_pts | 2.0767386573258126 |
| alerts_per_quarter_t90 | 20.5 |
| alert_precision_t90 | 0.6341463414634146 |
| accounts_scored_now | 448.0 |
| high_risk_accounts_now | 41.0 |
| high_risk_arr_now_jpy | 119810000.0 |
| high_risk_arr_renewing_180d_jpy | 96200000.0 |
| hidden_risk_accounts_now | 36.0 |
| hidden_risk_arr_now_jpy | 101870000.0 |
| expected_arr_at_risk

In [7]:
spark.createDataFrame(pd.DataFrame([(k, float(v)) for k, v in impact.items()], columns=["kpi", "value"])) \
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.gold_business_impact")
spark.sql(f"COMMENT ON TABLE {FQ}.gold_business_impact IS 'Business case KPIs (JPY) derived from out-of-time model evaluation and trailing-12-month churn. Save-rate assumptions are rows save_rate_early / save_rate_late.'")

print(f"""
Trailing 12 months: {t12.churned_logos_12m} logos churned, ¥{churned_arr/1e6:,.0f}M ARR lost, GRR {grr_now:.1%}
Status quo (react to cancellation ticket): saves ¥{saved_status_quo/1e6:,.1f}M/yr
With ML early warning at T-90:            saves ¥{saved_ml/1e6:,.1f}M/yr  (+¥{(saved_ml-saved_status_quo)/1e6:,.1f}M, GRR +{impact['grr_uplift_pts']:.1f} pts)
Today: {now.high} high-risk accounts, ¥{now.high_arr/1e6:,.0f}M ARR; {now.hidden} of them NOT flagged by CSMs (¥{now.hidden_arr/1e6:,.0f}M)
""")


Trailing 12 months: 93 logos churned, ¥330M ARR lost, GRR 84.6%
Status quo (react to cancellation ticket): saves ¥15.4M/yr
With ML early warning at T-90:            saves ¥60.0M/yr  (+¥44.5M, GRR +2.1 pts)
Today: 41 high-risk accounts, ¥120M ARR; 36 of them NOT flagged by CSMs (¥102M)



In [8]:
dbutils.notebook.exit(json.dumps({k: round(v, 4) for k, v in impact.items()}))

{"arr_live_now_jpy": 2190690000.0, "arr_start_12m_jpy": 2144320000.0, "churned_logos_12m": 93.0, "churned_arr_12m_jpy": 329910000.0, "gross_revenue_retention_12m": 0.8461, "ml_arr_coverage_t90": 0.606, "rules_arr_coverage_t90": 0.5385, "cancel_ticket_coverage_any_time": 0.4681, "arr_saved_status_quo_jpy": 15442595.7447, "arr_saved_with_ml_jpy": 59974518.1214, "incremental_arr_saved_vs_status_quo_jpy": 44531922.3768, "incremental_arr_saved_vs_rules_jpy": 6675273.1419, "grr_with_ml": 0.8669, "grr_uplift_pts": 2.0767, "alerts_per_quarter_t90": 20.5, "alert_precision_t90": 0.6341, "accounts_scored_now": 448.0, "high_risk_accounts_now": 41.0, "high_risk_arr_now_jpy": 119810000.0, "high_risk_arr_renewing_180d_jpy": 96200000.0, "hidden_risk_accounts_now": 36.0, "hidden_risk_arr_now_jpy": 101870000.0, "expected_arr_at_risk_now_jpy": 280038415.0, "save_rate_early": 0.3, "save_rate_late": 0.1}